In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("data/qoute_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.shape

(3038, 2)

In [5]:
quotes = df['quote']

In [6]:
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [7]:
quotes = quotes.str.lower()

In [8]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices, harry, that show what we t...
2    “there are only two ways to live your life. on...
3    “the person, be it gentleman or lady, who has ...
4    “imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [9]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [10]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: str

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [12]:
word_index = tokenizer.word_index
len(word_index)

8978

In [13]:
list(word_index.items())[:10]

[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [14]:
sequence = tokenizer.texts_to_sequences(quotes)

In [15]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [16]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [17]:
X = []
y = []

for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [18]:
X[:10]

[[713],
 [713, 62],
 [713, 62, 29],
 [713, 62, 29, 19],
 [713, 62, 29, 19, 16],
 [713, 62, 29, 19, 16, 946],
 [713, 62, 29, 19, 16, 946, 10],
 [713, 62, 29, 19, 16, 946, 10, 7],
 [713, 62, 29, 19, 16, 946, 10, 7, 5],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156]]

In [19]:
len(X)

85271

In [20]:
len(y)

85271

In [21]:
y[:10]

[62, 29, 19, 16, 946, 10, 7, 5, 1156, 8]

In [22]:
max_len = max(len(x) for x in X)
max_len

745

In [23]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [24]:
X_padded = pad_sequences(X, maxlen=max_len, padding="pre") 

In [25]:
X_padded[:10]

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,  946,   10,    7],
       [   0,    0,    0, ...,   10,    7,    5],
       [   0,    0,    0, ...,    7,    5, 1156]],
      shape=(10, 745), dtype=int32)

In [26]:
y = np.array(y)

In [27]:
X_padded.shape

(85271, 745)

In [28]:
y.shape

(85271,)

In [29]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes = vocab_size)

In [30]:
y_one_hot.shape

(85271, 10000)

In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Embedding

In [32]:
embedding_dim = 256
rnn_units = 512

In [33]:
rnn_model = Sequential()
rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim))
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation="softmax"))

In [34]:
rnn_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [35]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim))
lstm_model.add(SimpleRNN(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation="softmax"))

In [37]:
lstm_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [38]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_1 (SimpleRNN)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [39]:
epochs = 10
batch_size = 512

In [40]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 428s 3s/step - accuracy: 0.0340 - loss: 6.9471 - val_accuracy: 0.0376 - val_loss: 6.8205
Epoch 2/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 452s 3s/step - accuracy: 0.0575 - loss: 6.4031 - val_accuracy: 0.0750 - val_loss: 6.6138
Epoch 3/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 499s 3s/step - accuracy: 0.0604 - loss: 7.0405 - val_accuracy: 0.0725 - val_loss: 6.7493
Epoch 4/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 498s 3s/step - accuracy: 0.0872 - loss: 6.2216 - val_accuracy: 0.0847 - val_loss: 6.4889
Epoch 5/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 498s 3s/step - accuracy: 0.0959 - loss: 6.0114 - val_accuracy: 0.0929 - val_loss: 6.4862
Epoch 6/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 496s 3s/step - accuracy: 0.1028 - loss: 5.8323 - val_accuracy: 0.0955 - val_loss: 6.5106
Epoch 7/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 499s 3s/step - accuracy: 0.1094 - loss: 5.7614 - val_accuracy: 0.0999 - val_loss: 6.4909
Epoch 8/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 504s 3s/step - accuracy: 0.1137 - loss: 5.7019 - val_accu

In [42]:
rnn_model.save("rnn_model.h5")

In [ ]:
history_lstm = lstm_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

In [ ]:
lstm_model.save("lstm_model_1.h5")

In [43]:
index_to_word = {}
for word, index in word_index.items():
    index_to_word[index] = word

In [52]:
def predictor(model, tokenizer, text, max_len):
    text = text.lower()
    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_len, padding="pre")
    pred = model.predict(seq, verbose=0)
    pred_index = np.argmax(pred)
    return index_to_word[pred_index]

In [53]:
import tensorflow as tf
lstm_model = tf.keras.models.load_model('lstm_model_1.h5')

In [71]:
seed_text = "I think my love of my life is"
next_word = predictor(lstm_model, tokenizer, seed_text, max_len)
next_word

'never'

In [72]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [74]:
seed = "the meaning of life"
generated_text = generate_text(lstm_model,tokenizer,seed,max_len,20)
print(generated_text)

the meaning of life is a place of life is akin to choosing immobility as a means of transportation than you can give you
